# Inspect MFCAD segmentation predictions

Loads the accepted model (`outputs/final/mfcad_segmentation_head.pt` -- MLP-128 head, class-weighted loss, best checkpoint epoch 36/40, val_acc=0.6035) and lets you inspect its per-face predictions on any validation-set file: colored 3D views (predicted vs. true class) plus a per-face table.

Requires the `brepnet` conda kernel (occwl / pythonocc-core / pytorch / pythreejs). If face coloring doesn't render, check the kernel selected in the top-right is "Python (brepnet)".

**Pattern:** run the setup cell once, then re-run the "pick a file" cell (and the display cells below it) as many times as you like, changing `FILE_TO_INSPECT`.

## Setup (run once)

In [ ]:
import argparse
import json
import sys
from pathlib import Path

import numpy as np
import torch
from matplotlib import pyplot as plt

# Project layout: this notebook lives in notebooks/, one level below project root.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BREPNET_DIR = PROJECT_ROOT / "external" / "BRepNet"
OCCWL_DIR = PROJECT_ROOT / "external" / "occwl" / "src"
sys.path.insert(0, str(BREPNET_DIR))
sys.path.insert(0, str(OCCWL_DIR))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from data.mfcad_loader import FEAT_NAMES, load_labeled_sample
from model.brepnet_backbone import load_frozen_backbone_with_new_head
from dataloaders.brepnet_dataset import BRepNetDataset, brepnet_collate_fn
from occwl.jupyter_viewer import JupyterViewer

NUM_CLASSES = len(FEAT_NAMES)
STEP_DIR = PROJECT_ROOT / "data" / "MFCAD" / "dataset" / "step"
FEATURES_DIR = PROJECT_ROOT / "outputs" / "brepnet_features_1000"
LABELS_CACHE_DIR = PROJECT_ROOT / "outputs" / "mfcad_labels_cache_1000"
SPLIT_PATH = PROJECT_ROOT / "outputs" / "mfcad_train_val_split.json"
CHECKPOINT_PATH = PROJECT_ROOT / "outputs" / "final" / "mfcad_segmentation_head.pt"
KERNEL_PATH = BREPNET_DIR / "kernels" / "winged_edge.json"
INPUT_FEATURES_PATH = BREPNET_DIR / "feature_lists" / "all.json"

# One visually-distinct color per class (tab20 is a fixed 20-color categorical
# palette -- distinct colors matter here since these are discrete class labels,
# not a continuous scalar, so a rainbow/heatmap colormap would be misleading).
cmap = plt.get_cmap("tab20")
CLASS_COLORS = [
    "#%02x%02x%02x" % tuple(int(c * 255) for c in cmap(i)[:3]) for i in range(NUM_CLASSES)
]

# Load the accepted model. hidden_size/dropout are read from the checkpoint's
# own saved metadata, not assumed, matching scripts/report_per_class_stats.py.
checkpoint = torch.load(CHECKPOINT_PATH, weights_only=False)
model = load_frozen_backbone_with_new_head(
    NUM_CLASSES,
    hidden_size=checkpoint["hidden_size"],
    dropout=checkpoint.get("dropout"),
)
model.classification_layer.load_state_dict(checkpoint["classification_layer_state_dict"])
model.eval()
print(f"Loaded checkpoint: epoch {checkpoint['saved_epoch']}/40, val_acc={checkpoint['saved_epoch_val_acc']:.4f}")

# The val-set file list (same split used to compute the accepted result).
split = json.loads(SPLIT_PATH.read_text())
val_stems = split["val_set"]
print(f"Val set: {len(val_stems)} files")

# BRepNetDataset wired to the val split -- reused unmodified (same as
# scripts/train_head.py / scripts/report_per_class_stats.py) so its
# already-verified small/big-face permutation logic handles batching for us.
opts = argparse.Namespace(
    dataset_dir=str(FEATURES_DIR),
    dataset_file=str(SPLIT_PATH),
    label_dir=str(LABELS_CACHE_DIR),
    kernel=str(KERNEL_PATH),
    input_features=str(INPUT_FEATURES_PATH),
)
val_dataset = BRepNetDataset(opts, "val_set")
print("Setup complete.")

## Pick a file to inspect (edit and re-run from here down)

In [ ]:
# EDIT THIS ONE LINE: an integer index into val_stems, or a filename stem string.
FILE_TO_INSPECT = 0  # e.g. try 42, or "0-0-2-7-7-23"

if isinstance(FILE_TO_INSPECT, int):
    file_stem = val_stems[FILE_TO_INSPECT]
else:
    file_stem = FILE_TO_INSPECT
    assert file_stem in val_stems, f"{file_stem} is not in the val set"
print(f"Inspecting: {file_stem}")

# True faces + labels, in occwl's raw traversal order -- proven in Phase 1/2
# to match the order the extracted npz features (and hence the model's
# predictions, once un-permuted below) are in.
sample = load_labeled_sample(STEP_DIR / f"{file_stem}.step")
true_labels = np.array(sample.labels)

# Real inference through the accepted model, same forward path as every
# training/eval script in this project (create_face_embeddings directly --
# never model.forward(), which is broken/unused upstream code).
idx = val_stems.index(file_stem)
item = val_dataset[idx]  # includes old_to_new_face_indices for this solid
batch = brepnet_collate_fn([item])
with torch.no_grad():
    face_embeddings = model.create_face_embeddings(
        batch["face_features"], batch["face_point_grids"],
        batch["edge_features"], batch["edge_point_grids"],
        batch["coedge_features"], batch["coedge_point_grids"],
        batch["face_kernel_tensor"], batch["edge_kernel_tensor"], batch["coedge_kernel_tensor"],
        batch["coedges_of_edges"], batch["coedges_of_small_faces"], batch["coedges_of_big_faces"],
    )
    logits_batch_order = model.classification_layer(face_embeddings)
preds_batch_order = logits_batch_order.argmax(dim=1).numpy()

# Gather predictions back into occwl/npz order (same order as sample.faces /
# true_labels) using the same verified permutation as
# scripts/report_per_class_stats.py -- not assumed to be identity, even
# though it happens to be for every file in this dataset (see
# notebooks/verify_dataset_face_permutation.py).
old_to_new = item["old_to_new_face_indices"].numpy()
pred_labels = preds_batch_order[old_to_new]

assert len(pred_labels) == len(true_labels) == len(sample.faces)
n_correct = int((pred_labels == true_labels).sum())
accuracy = n_correct / len(true_labels)
print(f"File accuracy: {n_correct}/{len(true_labels)} = {accuracy:.4f}")

### Predicted classes (colored 3D view)

In [ ]:
viewer_pred = JupyterViewer()
for face, cls in zip(sample.faces, pred_labels):
    viewer_pred.display(face, shape_color=CLASS_COLORS[cls], render_edges=True, edge_color="BLACK")
viewer_pred.show()

### True classes (colored 3D view, for comparison)

In [ ]:
viewer_true = JupyterViewer()
for face, cls in zip(sample.faces, true_labels):
    viewer_true.display(face, shape_color=CLASS_COLORS[cls], render_edges=True, edge_color="BLACK")
viewer_true.show()

### Per-face table and class-color legend

In [ ]:
print(f"{'face':>4}  {'true':<24} {'pred':<24} correct")
for i, (t, p) in enumerate(zip(true_labels, pred_labels)):
    mark = "OK" if t == p else "WRONG"
    print(f"{i:>4}  {FEAT_NAMES[t]:<24} {FEAT_NAMES[p]:<24} {mark}")

print("\nLegend:")
for i, name in enumerate(FEAT_NAMES):
    print(f"  {CLASS_COLORS[i]}  [{i:2d}] {name}")